In [21]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
%pip install python-dotenv

from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


True

In [22]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
)

In [23]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [24]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [25]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [26]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [27]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'I was going to tell you a joke about pizza, but it’s a little too cheesy... \n\nAnd honestly, the delivery is terrible.',
   'extras': {'signature': 'EqIOCp8OAWkUfRMubOCSQKZrSRWF8woo/Nz/Xamns2wMJVpTqzPhkSG6q02q/Q0UdLdlVQgfnOiO1p2gj6B7amaGwyGpqMSpWrM5voFn9u/U1a1tI5X4W+e9DZhtp9bwqpi0urRKPoJBJr/hOguuzZTb19K1mdUV0PlPQNw+MrSRwQPTVnpbOb7NiivGtgU9kByFrpLEoabJlbA3f9R1vD31W43oJ+4k0ZO5HpD9uzUPQFGmHh3LX2zu/FRC7/NO+LfB+xTVcmqJw8dpw1aqlWinwuozNa6qAbO5U6lWQ/AcV4kpDZxBq25ZvLa7bivH8eVHbTz5ldKboXbf8VTbIdUYYPjfQHZzw/6fohHqH9RUwvNj7gQCTwGHLyvzn/wVCRYXelY5+e6AjXh2/DL3UCyUHeMfSFqjIlDXoR1tBvHSpbJGqlSU3ctOtqu2YP8GwF6CUCCBeZ7Ja03h2tdTvti1JQO332bvWtjhrmP8rIQYLhlldlbxK3Upet7zA4xEqbDlJNBRNEVRK2HwqyfNlB1WJW+fWBVttyoIYJs4ElEXycna6HsZHe2oLUr6WFacw1AQJ/uX3Jft18+gI+p3Gd4243ewkHSzvKqwg0v9ejg9EAoMmY8ROfmc6Fx6iKhTQg50+rAuBacQWauaCic9NRYCp61cdP25lcSp9S4D5lxljaPZdEzR9n6hqlBV6OPtlH+5RJr4DI9n5YOn2o8rOFPzERBqqRP01sdiSV2y2bSMG5oTLTpLgdBbi8Doc2dovVobMum3hxCEBjSR6fRBQjcP

In [28]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I was going to tell you a joke about pizza, but it’s a little too cheesy... \n\nAnd honestly, the delivery is terrible.', 'extras': {'signature': 'EqIOCp8OAWkUfRMubOCSQKZrSRWF8woo/Nz/Xamns2wMJVpTqzPhkSG6q02q/Q0UdLdlVQgfnOiO1p2gj6B7amaGwyGpqMSpWrM5voFn9u/U1a1tI5X4W+e9DZhtp9bwqpi0urRKPoJBJr/hOguuzZTb19K1mdUV0PlPQNw+MrSRwQPTVnpbOb7NiivGtgU9kByFrpLEoabJlbA3f9R1vD31W43oJ+4k0ZO5HpD9uzUPQFGmHh3LX2zu/FRC7/NO+LfB+xTVcmqJw8dpw1aqlWinwuozNa6qAbO5U6lWQ/AcV4kpDZxBq25ZvLa7bivH8eVHbTz5ldKboXbf8VTbIdUYYPjfQHZzw/6fohHqH9RUwvNj7gQCTwGHLyvzn/wVCRYXelY5+e6AjXh2/DL3UCyUHeMfSFqjIlDXoR1tBvHSpbJGqlSU3ctOtqu2YP8GwF6CUCCBeZ7Ja03h2tdTvti1JQO332bvWtjhrmP8rIQYLhlldlbxK3Upet7zA4xEqbDlJNBRNEVRK2HwqyfNlB1WJW+fWBVttyoIYJs4ElEXycna6HsZHe2oLUr6WFacw1AQJ/uX3Jft18+gI+p3Gd4243ewkHSzvKqwg0v9ejg9EAoMmY8ROfmc6Fx6iKhTQg50+rAuBacQWauaCic9NRYCp61cdP25lcSp9S4D5lxljaPZdEzR9n6hqlBV6OPtlH+5RJr4DI9n5YOn2o8rOFPzERBqqRP01sdiSV2y2bSMG5oTLTpLgdBbi8Doc2dovVobMum3hx

In [29]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I was going to tell you a joke about pizza, but it’s a little too cheesy... \n\nAnd honestly, the delivery is terrible.', 'extras': {'signature': 'EqIOCp8OAWkUfRMubOCSQKZrSRWF8woo/Nz/Xamns2wMJVpTqzPhkSG6q02q/Q0UdLdlVQgfnOiO1p2gj6B7amaGwyGpqMSpWrM5voFn9u/U1a1tI5X4W+e9DZhtp9bwqpi0urRKPoJBJr/hOguuzZTb19K1mdUV0PlPQNw+MrSRwQPTVnpbOb7NiivGtgU9kByFrpLEoabJlbA3f9R1vD31W43oJ+4k0ZO5HpD9uzUPQFGmHh3LX2zu/FRC7/NO+LfB+xTVcmqJw8dpw1aqlWinwuozNa6qAbO5U6lWQ/AcV4kpDZxBq25ZvLa7bivH8eVHbTz5ldKboXbf8VTbIdUYYPjfQHZzw/6fohHqH9RUwvNj7gQCTwGHLyvzn/wVCRYXelY5+e6AjXh2/DL3UCyUHeMfSFqjIlDXoR1tBvHSpbJGqlSU3ctOtqu2YP8GwF6CUCCBeZ7Ja03h2tdTvti1JQO332bvWtjhrmP8rIQYLhlldlbxK3Upet7zA4xEqbDlJNBRNEVRK2HwqyfNlB1WJW+fWBVttyoIYJs4ElEXycna6HsZHe2oLUr6WFacw1AQJ/uX3Jft18+gI+p3Gd4243ewkHSzvKqwg0v9ejg9EAoMmY8ROfmc6Fx6iKhTQg50+rAuBacQWauaCic9NRYCp61cdP25lcSp9S4D5lxljaPZdEzR9n6hqlBV6OPtlH+5RJr4DI9n5YOn2o8rOFPzERBqqRP01sdiSV2y2bSMG5oTLTpLgdBbi8Doc2dovVobMum3hx

In [31]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1b521d-d48d-6ea9-8001-6122dd96b04c"}})

{'topic': 'pizza',
 'joke': [{'type': 'text',
   'text': 'I was going to tell you a joke about pizza, but it’s a little too cheesy... \n\nAnd honestly, the delivery is terrible.',
   'extras': {'signature': 'EqIOCp8OAWkUfRMubOCSQKZrSRWF8woo/Nz/Xamns2wMJVpTqzPhkSG6q02q/Q0UdLdlVQgfnOiO1p2gj6B7amaGwyGpqMSpWrM5voFn9u/U1a1tI5X4W+e9DZhtp9bwqpi0urRKPoJBJr/hOguuzZTb19K1mdUV0PlPQNw+MrSRwQPTVnpbOb7NiivGtgU9kByFrpLEoabJlbA3f9R1vD31W43oJ+4k0ZO5HpD9uzUPQFGmHh3LX2zu/FRC7/NO+LfB+xTVcmqJw8dpw1aqlWinwuozNa6qAbO5U6lWQ/AcV4kpDZxBq25ZvLa7bivH8eVHbTz5ldKboXbf8VTbIdUYYPjfQHZzw/6fohHqH9RUwvNj7gQCTwGHLyvzn/wVCRYXelY5+e6AjXh2/DL3UCyUHeMfSFqjIlDXoR1tBvHSpbJGqlSU3ctOtqu2YP8GwF6CUCCBeZ7Ja03h2tdTvti1JQO332bvWtjhrmP8rIQYLhlldlbxK3Upet7zA4xEqbDlJNBRNEVRK2HwqyfNlB1WJW+fWBVttyoIYJs4ElEXycna6HsZHe2oLUr6WFacw1AQJ/uX3Jft18+gI+p3Gd4243ewkHSzvKqwg0v9ejg9EAoMmY8ROfmc6Fx6iKhTQg50+rAuBacQWauaCic9NRYCp61cdP25lcSp9S4D5lxljaPZdEzR9n6hqlBV6OPtlH+5RJr4DI9n5YOn2o8rOFPzERBqqRP01sdiSV2y2bSMG5oTLTpLgdBbi8Doc2dovVobMum3hxCEBjSR6fRBQjcP

In [37]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id":  "1f1b521d-d48d-6ea9-8001-6122dd96b04c", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1b5223-f131-6a6c-8002-29309ffd7acc'}}

In [39]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1b5223-f131-6a6c-8002-29309ffd7acc"}})

{'topic': 'samosa',
 'joke': [{'type': 'text',
   'text': 'I was going to tell you a joke about pizza, but it’s a little too cheesy... \n\nAnd honestly, the delivery is terrible.',
   'extras': {'signature': 'EqIOCp8OAWkUfRMubOCSQKZrSRWF8woo/Nz/Xamns2wMJVpTqzPhkSG6q02q/Q0UdLdlVQgfnOiO1p2gj6B7amaGwyGpqMSpWrM5voFn9u/U1a1tI5X4W+e9DZhtp9bwqpi0urRKPoJBJr/hOguuzZTb19K1mdUV0PlPQNw+MrSRwQPTVnpbOb7NiivGtgU9kByFrpLEoabJlbA3f9R1vD31W43oJ+4k0ZO5HpD9uzUPQFGmHh3LX2zu/FRC7/NO+LfB+xTVcmqJw8dpw1aqlWinwuozNa6qAbO5U6lWQ/AcV4kpDZxBq25ZvLa7bivH8eVHbTz5ldKboXbf8VTbIdUYYPjfQHZzw/6fohHqH9RUwvNj7gQCTwGHLyvzn/wVCRYXelY5+e6AjXh2/DL3UCyUHeMfSFqjIlDXoR1tBvHSpbJGqlSU3ctOtqu2YP8GwF6CUCCBeZ7Ja03h2tdTvti1JQO332bvWtjhrmP8rIQYLhlldlbxK3Upet7zA4xEqbDlJNBRNEVRK2HwqyfNlB1WJW+fWBVttyoIYJs4ElEXycna6HsZHe2oLUr6WFacw1AQJ/uX3Jft18+gI+p3Gd4243ewkHSzvKqwg0v9ejg9EAoMmY8ROfmc6Fx6iKhTQg50+rAuBacQWauaCic9NRYCp61cdP25lcSp9S4D5lxljaPZdEzR9n6hqlBV6OPtlH+5RJr4DI9n5YOn2o8rOFPzERBqqRP01sdiSV2y2bSMG5oTLTpLgdBbi8Doc2dovVobMum3hxCEBjSR6fRBQjc